# 2.2 Deploy Qwen3-4B-Instruct to SageMaker with LMI — Normal & Multi-LoRA

This notebook shows how to:
1. Deploy **Qwen3-4B-Instruct** to a SageMaker real-time endpoint using the [Large Model Inference (LMI)](https://docs.djl.ai/master/docs/serving/serving/docs/lmi/index.html) container with vLLM.
2. Fine-tune a QLoRA adapter with SageMaker Training and save it to S3.
3. Register the adapter as a SageMaker **Inference Component (IC)** for multi-LoRA serving.
4. Compare base model and fine-tuned adapter responses side-by-side.
5. Clean up all resources.

Uses the **SageMaker Python SDK v3** (`sagemaker.core`) throughout.

> **Instance used:** `ml.g5.2xlarge` (1× A10G, 24 GB VRAM). Qwen3-4B fits comfortably in bf16.

---


## 0. Prerequisites

Install required packages. Restart the kernel after this cell completes.


In [ ]:
%pip install -qU sagemaker==3.4.0 datasets==3.2.0

## 1. Imports and AWS clients

In [ ]:
import json
import os
import shutil
import time

import boto3
from IPython.display import Markdown, clear_output, display

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.resources import Endpoint, EndpointConfig, Model
from sagemaker.core.shapes import (
    ContainerDefinition,
    ProductionVariant,
    ProductionVariantManagedInstanceScaling,
    ProductionVariantRoutingConfig,
)
from sagemaker.core.helper.session_helper import _wait_until, _deploy_done

sagemaker_session = Session()
region            = sagemaker_session.boto_session.region_name
sm_runtime        = boto3.client("sagemaker-runtime")
s3_client         = boto3.client("s3")

print(f"Region: {region}")


## 2. Helper functions


In [ ]:
def get_last_job_name(job_name_prefix: str) -> str:
    """Return the most recent *completed* training job whose name starts with job_name_prefix."""
    sm_client = sagemaker_session.sagemaker_client
    matching_jobs = []
    next_token = None
    while True:
        params = {
            "Resource": "TrainingJob",
            "SearchExpression": {
                "Filters": [
                    {"Name": "TrainingJobName", "Operator": "Contains", "Value": job_name_prefix},
                    {"Name": "TrainingJobStatus", "Operator": "Equals",   "Value": "Completed"},
                ]
            },
            "SortBy": "CreationTime",
            "SortOrder": "Descending",
            "MaxResults": 100,
        }
        if next_token:
            params["NextToken"] = next_token
        resp = sm_client.search(**params)
        matching_jobs.extend(
            j["TrainingJob"]["TrainingJobName"]
            for j in resp["Results"]
            if j["TrainingJob"]["TrainingJobName"].startswith(job_name_prefix)
        )
        next_token = resp.get("NextToken")
        if not next_token or matching_jobs:
            break
    if not matching_jobs:
        raise ValueError(f"No completed training jobs found with prefix '{job_name_prefix}'")
    return matching_jobs[0]


## 3. Configuration

Role and bucket are auto-detected from the current SageMaker execution environment.


In [ ]:
# ── Identity & storage ────────────────────────────────────────────────────────
role           = get_execution_role(sagemaker_session, use_default=True)
bucket         = sagemaker_session.default_bucket()
default_prefix = (sagemaker_session.default_bucket_prefix or "").strip("/")

print(f"Role  : {role}")
print(f"Bucket: {bucket}")


In [ ]:
# ── Model & container ─────────────────────────────────────────────────────────
# Latest LMI image (vLLM backend, CUDA 12.8)
# See: https://docs.djl.ai/master/docs/serving/serving/docs/lmi/release_notes.html
LMI_IMAGE = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:0.36.0-lmi20.0.0-cu128-v1.0"

MODEL_ID  = "Qwen/Qwen3-4B-Instruct-2507"   # base model on HuggingFace Hub
INSTANCE  = "ml.g5.2xlarge"                 # 1× A10G — sufficient for 4B in bf16
TIMEOUT   = 600                             # container startup health-check timeout (s)

# Unique name prefix for all resources created in this notebook
RUN_ID        = time.strftime("%y%m%d-%H%M%S")
RESOURCE_NAME = f"qwen3-4b-{RUN_ID}"

print(f"LMI image : {LMI_IMAGE}")
print(f"Model     : {MODEL_ID}")
print(f"Instance  : {INSTANCE}")
print(f"Run ID    : {RUN_ID}")

---
## Part A — Deploy the base model

We create a SageMaker **Model**, **EndpointConfig**, and **Endpoint** using SDK v3 resource
classes, backed by the LMI container with vLLM.

Key LMI environment variables:
| Variable | Value | Purpose |
|---|---|---|
| `HF_MODEL_ID` | `Qwen/Qwen3-4B-Instruct-2507` | Model to download from HF Hub |
| `OPTION_ENTRYPOINT` | `djl_python.lmi_vllm.vllm_async_service` | vLLM async handler |
| `OPTION_ASYNC_MODE` | `true` | Enable async request processing |
| `OPTION_ROLLING_BATCH` | `disable` | Required when async mode is on |
| `OPTION_DTYPE` | `bf16` | Weight precision |
| `OPTION_MAX_MODEL_LEN` | `8192` | Max context window (tokens) |
| `OPTION_ENABLE_LORA` | `true` | Enable LoRA adapter support |
| `OPTION_MAX_LORAS` | `4` | Max adapters loaded on GPU simultaneously |
| `OPTION_MAX_CPU_LORAS` | `8` | Max adapters cached in CPU memory |
| `OPTION_MAX_LORA_RANK` | `64` | Max LoRA rank supported |


### 3.1 Create SageMaker Model

In [ ]:
model_name = RESOURCE_NAME

env = {
    # Model source
    "HF_MODEL_ID": MODEL_ID,
    # vLLM async handler
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ROLLING_BATCH": "disable",
    # Precision & context
    "OPTION_DTYPE": "bf16",
    "OPTION_MAX_MODEL_LEN": "8192",
    "OPTION_TRUST_REMOTE_CODE": "true",
    # LoRA multi-adapter support
    "OPTION_ENABLE_LORA": "true",
    "OPTION_MAX_LORAS": "4",
    "OPTION_MAX_CPU_LORAS": "8",
    "OPTION_MAX_LORA_RANK": "64",
    # Fail fast on startup errors
    "SERVING_FAIL_FAST": "true",
}

core_model = Model.create(
    model_name=model_name,
    execution_role_arn=role,
    primary_container=ContainerDefinition(
        image=LMI_IMAGE,
        environment=env,
    ),
)
print(f"Model created: {model_name}")


### 3.2 Create Endpoint Config and Endpoint

For Inference Component-based endpoints the `ProductionVariant` provisions **bare compute** only —
no `model_name` is set here. Models are attached at the IC level in sections 3.3 and B.4.

We size the variant to hold **two ICs** (base model + LoRA adapter):
- `managed_instance_scaling` with `min=1, max=1` pins the instance count
- `routing_config` with `RoutingStrategy=LEAST_OUTSTANDING_REQUESTS` distributes traffic across ICs


In [ ]:
endpoint_config_name = RESOURCE_NAME
endpoint_name        = RESOURCE_NAME
variant_name         = "AllTraffic"

# Two ICs will share this instance: base model (1 GPU) + LoRA adapter (piggybacks base IC)
# ProductionVariant has NO model_name — models are attached via Inference Components.
# execution_role_arn is required when Inference Components are enabled.
EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    execution_role_arn=role,
    production_variants=[
        ProductionVariant(
            variant_name=variant_name,
            instance_type=INSTANCE,
            initial_instance_count=1,
            managed_instance_scaling=ProductionVariantManagedInstanceScaling(
                status="ENABLED",
                min_instance_count=1,
                max_instance_count=1,
            ),
            routing_config=ProductionVariantRoutingConfig(
                routing_strategy="LEAST_OUTSTANDING_REQUESTS",
            ),
            container_startup_health_check_timeout_in_seconds=TIMEOUT,
        )
    ],
)

core_endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)

_wait_until(lambda: _deploy_done(sagemaker_session.sagemaker_client, endpoint_name), poll=30)
core_endpoint = Endpoint.get(endpoint_name=endpoint_name)
print(f"Endpoint status: {core_endpoint.endpoint_status}")


### 3.3 Create base Inference Component

An **Inference Component (IC)** maps a model to a slice of the endpoint's compute.
The base IC hosts the Qwen3-4B weights; adapter ICs will reference it.

In [ ]:
base_ic_name = f"base-{RESOURCE_NAME}"
sm_client    = sagemaker_session.sagemaker_client

sm_client.create_inference_component(
    InferenceComponentName=base_ic_name,
    EndpointName=endpoint_name,
    VariantName=variant_name,
    Specification={
        "ModelName": model_name,
        "StartupParameters": {
            "ModelDataDownloadTimeoutInSeconds": TIMEOUT,
            "ContainerStartupHealthCheckTimeoutInSeconds": TIMEOUT,
        },
        "ComputeResourceRequirements": {
            "MinMemoryRequiredInMb": 8192,
            "NumberOfAcceleratorDevicesRequired": 1,
        },
    },
    RuntimeConfig={"CopyCount": 1},
)

def wait_for_ic(ic_name: str, poll: int = 30) -> str:
    """Block until the Inference Component leaves Creating state."""
    progress = f"Waiting for IC '{ic_name}'"
    print(progress)
    status = sm_client.describe_inference_component(
        InferenceComponentName=ic_name
    )["InferenceComponentStatus"]
    while status == "Creating":
        time.sleep(poll)
        status = sm_client.describe_inference_component(
            InferenceComponentName=ic_name
        )["InferenceComponentStatus"]
        clear_output(wait=True)
        progress += "."
        print(progress)
    print(f"IC '{ic_name}' \u2192 {status}")
    return status

wait_for_ic(base_ic_name)


### 3.4 Test the base model

In [ ]:
SYSTEM_PROMPT = (
    "You are a medical expert with advanced knowledge in clinical reasoning, "
    "diagnostics, and treatment planning. "
    "Before answering, think carefully and create a step-by-step chain of thoughts."
)

USER_PROMPT = (
    "A 3-week-old child has been diagnosed with late-onset perinatal meningitis "
    "and the CSF culture shows gram-positive bacilli. "
    "What characteristic of this bacterium specifically differentiates it from other agents?"
)

payload = json.dumps({
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT},
    ],
    "max_tokens": 512,
    "temperature": 0.2,
})

t0 = time.time()
res = core_endpoint.invoke(
    body=payload,
    content_type="application/json",
    accept="application/json",
    inference_component_name=base_ic_name,
)
response = json.loads(res.body.read().decode("utf-8"))
print(f"Response time: {time.time()-t0:.2f}s")
display(Markdown(response["choices"][0]["message"]["content"]))
print(f"\nUsage: {response['usage']}")


---
## Part B — Fine-tune a LoRA adapter and serve it with Inference Components

This section runs a QLoRA fine-tuning job on SageMaker Training, then registers the
resulting adapter as an Inference Component on the endpoint deployed in Part A.

Steps:
1. Prepare and upload the training dataset to S3
2. Launch a SageMaker Training job (QLoRA, `merge_weights: false` → adapter-only output)
3. Locate the adapter archive in S3
4. Register it as an Inference Component and test it


### B.1 Prepare and upload the training dataset

We use the [FreedomIntelligence/medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT) dataset,
the same one used in Lab 02.01.


In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are a medical expert with advanced knowledge in clinical reasoning, "
    "diagnostics, and treatment planning. "
    "Before answering, think carefully and create a step-by-step chain of thoughts."
)

def convert_to_messages(sample, system_prompt=""):
    sample["messages"] = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": sample["Question"]},
        {"role": "assistant", "content": f"{sample['Complex_CoT']}\n\n{sample['Response']}"},
    ]
    return sample

num_samples = 100
full_dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT", "en",
    split=f"train[:{num_samples}]"
)
splits       = full_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = splits["train"].map(
    convert_to_messages, remove_columns=list(full_dataset.features)
)
test_dataset  = splits["test"].map(
    convert_to_messages, remove_columns=list(full_dataset.features)
)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")


In [ ]:

if default_prefix:
    dataset_prefix = f"{default_prefix}/datasets/lmi-lora-sft"
else:
    dataset_prefix = "datasets/lmi-lora-sft"

os.makedirs("./data/train", exist_ok=True)
os.makedirs("./data/test",  exist_ok=True)
train_dataset.to_json("./data/train/dataset.json", orient="records")
test_dataset.to_json( "./data/test/dataset.json",  orient="records")

s3_client.upload_file("./data/train/dataset.json", bucket, f"{dataset_prefix}/train/dataset.json")
s3_client.upload_file("./data/test/dataset.json",  bucket, f"{dataset_prefix}/test/dataset.json")
shutil.rmtree("./data")

train_s3 = f"s3://{bucket}/{dataset_prefix}/train/dataset.json"
test_s3  = f"s3://{bucket}/{dataset_prefix}/test/dataset.json"
print(f"Train: {train_s3}")
print(f"Test : {test_s3}")


### B.2 Configure and launch the SageMaker Training job

Key difference from Lab 02.01: **`merge_weights: false`** so the output contains the raw
LoRA adapter files (`adapter_config.json`, `adapter_model.safetensors`) rather than a
full merged model. LMI requires unmerged adapter weights to serve via Inference Components.


In [ ]:
TRAIN_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

args_yaml = f"""model_id: "{TRAIN_MODEL_ID}"

# SageMaker paths (set by the training container)
output_dir: "/opt/ml/model"
train_dataset_path: "/opt/ml/input/data/train/"
test_dataset_path: "/opt/ml/input/data/test/"

# LoRA hyperparameters
max_seq_length: 1500
lora_r: 8
lora_alpha: 16
lora_dropout: 0.1

# Training hyperparameters
learning_rate: 2e-4
num_train_epochs: 1
per_device_train_batch_size: 2
per_device_eval_batch_size: 1
gradient_accumulation_steps: 2
gradient_checkpointing: true
fp16: true
bf16: false
tf32: false

# Keep adapter unmerged so LMI can load it as a LoRA IC
merge_weights: false
"""

with open("args.yaml", "w") as f:
    f.write(args_yaml)

print(args_yaml)


In [ ]:
from sagemaker.core.s3 import S3Uploader
from sagemaker.core import image_uris
from sagemaker.train.model_trainer import ModelTrainer, InputData, Torchrun, StoppingCondition
from sagemaker.core.training.configs import Compute, SourceCode
from sagemaker.core.shapes import OutputDataConfig

train_instance = "ml.g5.2xlarge"

# Upload training config to S3
if default_prefix:
    config_s3_prefix = f"s3://{bucket}/{default_prefix}/training_config/lmi-lora"
else:
    config_s3_prefix = f"s3://{bucket}/training_config/lmi-lora"

config_s3_path = S3Uploader.upload(
    local_path="args.yaml",
    desired_s3_uri=f"{config_s3_prefix}/config"
)
print(f"Config uploaded to: {config_s3_path}")

# PyTorch training image
training_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.6.0",
    instance_type=train_instance,
    image_scope="training",
)
print(f"Training image: {training_image}")


In [ ]:
train_job_prefix = f"train-{TRAIN_MODEL_ID.split('/')[-1].replace('.', '-')}-lora-lmi"

if default_prefix:
    train_output_path = f"s3://{bucket}/{default_prefix}/{train_job_prefix}"
else:
    train_output_path = f"s3://{bucket}/{train_job_prefix}"

model_trainer = ModelTrainer(
    training_image=training_image,
    source_code=SourceCode(
        source_dir="./scripts",
        requirements="requirements.txt",
        entry_script="train.py",
    ),
    base_job_name=train_job_prefix,
    compute=Compute(
        instance_type=train_instance,
        instance_count=1,
        keep_alive_period_in_seconds=3600,
        volume_size_in_gb=50,
    ),
    distributed=Torchrun(),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
    hyperparameters={"config": "/opt/ml/input/data/config/args.yaml"},
    output_data_config=OutputDataConfig(s3_output_path=train_output_path),
    environment={"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"},
)

model_trainer.train(
    input_data_config=[
        InputData(channel_name="train",  data_source=train_s3),
        InputData(channel_name="test",   data_source=test_s3),
        InputData(channel_name="config", data_source=config_s3_path),
    ],
    wait=True,
)


### B.3 Locate the adapter archive in S3

SageMaker writes training output to `<output_path>/<job_name>/output/model.tar.gz`.
We derive the URI from the completed job and verify it exists before registering.


In [ ]:
training_job_name = get_last_job_name(train_job_prefix)
print(f"Training job: {training_job_name}")

adapter_s3_uri = f"{train_output_path}/{training_job_name}/output/model.tar.gz"
print(f"Adapter S3 URI: {adapter_s3_uri}")

# Verify it exists
parsed = adapter_s3_uri.replace("s3://", "").split("/", 1)
head = s3_client.head_object(Bucket=parsed[0], Key=parsed[1])
print(f"\u2705 Archive found — {head['ContentLength'] / 1024 / 1024:.1f} MB")

%store adapter_s3_uri


### B.4 Register the adapter as an Inference Component

The adapter IC sets `BaseInferenceComponentName` to point at the base IC.
LMI loads the LoRA weights on top of the base model at request time.


In [ ]:
adapter_ic_name = f"adapter-{RESOURCE_NAME}"

sm_client.create_inference_component(
    InferenceComponentName=adapter_ic_name,
    EndpointName=endpoint_name,
    Specification={
        "BaseInferenceComponentName": base_ic_name,
        "Container": {
            "ArtifactUrl": adapter_s3_uri,
        },
    },
)

wait_for_ic(adapter_ic_name)


### B.5 Test the adapter IC


In [ ]:
t0 = time.time()
res_adapter = core_endpoint.invoke(
    body=payload,
    content_type="application/json",
    accept="application/json",
    inference_component_name=adapter_ic_name,
)
response_adapter = json.loads(res_adapter.body.read().decode("utf-8"))
print(f"Response time: {time.time()-t0:.2f}s")
display(Markdown(response_adapter["choices"][0]["message"]["content"]))
print(f"\nUsage: {response_adapter['usage']}")


### B.6 Side-by-side comparison


In [ ]:
print("=" * 60)
print("BASE MODEL")
print("=" * 60)
print(response["choices"][0]["message"]["content"][:500], "...")

print()
print("=" * 60)
print("ADAPTER MODEL")
print("=" * 60)
print(response_adapter["choices"][0]["message"]["content"][:500], "...")

---
## Part C — Second LoRA adapter: Financial Reasoning (FinQA)

We now fine-tune a **second** LoRA adapter on the
[FinQA](https://huggingface.co/datasets/gagan3012/finqa-updated) dataset — 6,251 financial
Q&A pairs drawn from real earnings reports, requiring numerical reasoning over structured
and unstructured evidence.

This adapter is registered as a **third Inference Component** on the same endpoint alongside
the base model (Part A) and the medical adapter (Part B). Each IC is independently
addressable at inference time — no redeployment needed.

Steps mirror Part B exactly, with a different dataset and system prompt.


### C.1 Prepare and upload the FinQA training dataset

FinQA records contain a `query` (question + financial context) and a numerical `answer`.
We format them into the standard chat template the training script expects.


In [ ]:
from datasets import load_dataset

FINQA_SYSTEM_PROMPT = (
    "You are a financial analyst with expertise in numerical reasoning over financial reports. "
    "Given a question and relevant context from financial documents, "
    "reason step-by-step and provide a precise numerical answer."
)

def convert_finqa_to_messages(sample):
    sample["messages"] = [
        {"role": "system",    "content": FINQA_SYSTEM_PROMPT},
        {"role": "user",      "content": sample["query"]},
        {"role": "assistant", "content": sample["answer"]},
    ]
    return sample

finqa_num_samples = 500  # ~8 % of the full training split — adjust as needed
finqa_full = load_dataset(
    "gagan3012/finqa-updated",
    split=f"train[:{finqa_num_samples}]",
)
finqa_splits       = finqa_full.train_test_split(test_size=0.1, seed=42)
finqa_train = finqa_splits["train"].map(
    convert_finqa_to_messages, remove_columns=list(finqa_full.features)
)
finqa_test  = finqa_splits["test"].map(
    convert_finqa_to_messages, remove_columns=list(finqa_full.features)
)

print(f"Train samples: {len(finqa_train)}, Test samples: {len(finqa_test)}")
print("\nSample message:")
print(finqa_train[0]["messages"][1]["content"][:300])


In [ ]:
import os, shutil

if default_prefix:
    finqa_dataset_prefix = f"{default_prefix}/datasets/lmi-lora-finqa"
else:
    finqa_dataset_prefix = "datasets/lmi-lora-finqa"

os.makedirs("./data/train", exist_ok=True)
os.makedirs("./data/test",  exist_ok=True)
finqa_train.to_json("./data/train/dataset.json", orient="records")
finqa_test.to_json( "./data/test/dataset.json",  orient="records")

s3_client.upload_file("./data/train/dataset.json", bucket, f"{finqa_dataset_prefix}/train/dataset.json")
s3_client.upload_file("./data/test/dataset.json",  bucket, f"{finqa_dataset_prefix}/test/dataset.json")
shutil.rmtree("./data")

finqa_train_s3 = f"s3://{bucket}/{finqa_dataset_prefix}/train/dataset.json"
finqa_test_s3  = f"s3://{bucket}/{finqa_dataset_prefix}/test/dataset.json"
print(f"Train: {finqa_train_s3}")
print(f"Test : {finqa_test_s3}")


### C.2 Configure and launch the SageMaker Training job

Same configuration as Part B — `merge_weights: false` to produce a standalone LoRA adapter.


In [ ]:
finqa_args_yaml = f"""model_id: "{TRAIN_MODEL_ID}"

# SageMaker paths (set by the training container)
output_dir: "/opt/ml/model"
train_dataset_path: "/opt/ml/input/data/train/"
test_dataset_path: "/opt/ml/input/data/test/"

# LoRA hyperparameters
max_seq_length: 1500
lora_r: 8
lora_alpha: 16
lora_dropout: 0.1

# Training hyperparameters
learning_rate: 2e-4
num_train_epochs: 1
per_device_train_batch_size: 2
per_device_eval_batch_size: 1
gradient_accumulation_steps: 2
gradient_checkpointing: true
fp16: true
bf16: false
tf32: false

# Keep adapter unmerged so LMI can load it as a LoRA IC
merge_weights: false
"""

with open("finqa_args.yaml", "w") as f:
    f.write(finqa_args_yaml)

print(finqa_args_yaml)


In [ ]:
from sagemaker.core.s3 import S3Uploader
from sagemaker.core import image_uris
from sagemaker.train.model_trainer import ModelTrainer, InputData, Torchrun, StoppingCondition
from sagemaker.core.training.configs import Compute, SourceCode
from sagemaker.core.shapes import OutputDataConfig

# Upload training config to S3
if default_prefix:
    finqa_config_s3_prefix = f"s3://{bucket}/{default_prefix}/training_config/lmi-lora-finqa"
else:
    finqa_config_s3_prefix = f"s3://{bucket}/training_config/lmi-lora-finqa"

finqa_config_s3_path = S3Uploader.upload(
    local_path="finqa_args.yaml",
    desired_s3_uri=f"{finqa_config_s3_prefix}/config",
)
print(f"Config uploaded to: {finqa_config_s3_path}")

training_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.6.0",
    instance_type="ml.g5.2xlarge",
    image_scope="training",
)
print(f"Training image: {training_image}")


In [ ]:
finqa_job_prefix = f"train-{TRAIN_MODEL_ID.split('/')[-1].replace('.', '-')}-lora-finqa"

if default_prefix:
    finqa_output_path = f"s3://{bucket}/{default_prefix}/{finqa_job_prefix}"
else:
    finqa_output_path = f"s3://{bucket}/{finqa_job_prefix}"

finqa_trainer = ModelTrainer(
    training_image=training_image,
    source_code=SourceCode(
        source_dir="./scripts",
        requirements="requirements.txt",
        entry_script="train.py",
    ),
    base_job_name=finqa_job_prefix,
    compute=Compute(
        instance_type="ml.g5.2xlarge",
        instance_count=1,
        keep_alive_period_in_seconds=3600,
        volume_size_in_gb=50,
    ),
    distributed=Torchrun(),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
    hyperparameters={"config": "/opt/ml/input/data/config/finqa_args.yaml"},
    output_data_config=OutputDataConfig(s3_output_path=finqa_output_path),
    environment={"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"},
)

finqa_trainer.train(
    input_data_config=[
        InputData(channel_name="train",  data_source=finqa_train_s3),
        InputData(channel_name="test",   data_source=finqa_test_s3),
        InputData(channel_name="config", data_source=finqa_config_s3_path),
    ],
    wait=True,
)


### C.3 Locate the FinQA adapter archive in S3


In [ ]:
finqa_job_name = get_last_job_name(finqa_job_prefix)
print(f"Training job: {finqa_job_name}")

finqa_adapter_s3_uri = f"{finqa_output_path}/{finqa_job_name}/output/model.tar.gz"
print(f"Adapter S3 URI: {finqa_adapter_s3_uri}")

# Verify it exists
parsed = finqa_adapter_s3_uri.replace("s3://", "").split("/", 1)
head = s3_client.head_object(Bucket=parsed[0], Key=parsed[1])
print(f"\u2705 Archive found — {head['ContentLength'] / 1024 / 1024:.1f} MB")

%store finqa_adapter_s3_uri


### C.4 Register the FinQA adapter as an Inference Component

Like the medical adapter in Part B, this IC sets `BaseInferenceComponentName` to the base IC.
The endpoint now hosts three independently addressable ICs:
- `base-{RESOURCE_NAME}` — base Qwen3-4B
- `adapter-{RESOURCE_NAME}` — medical reasoning adapter (Part B)
- `finqa-adapter-{RESOURCE_NAME}` — financial reasoning adapter (Part C)


In [ ]:
finqa_ic_name = f"finqa-adapter-{RESOURCE_NAME}"

sm_client.create_inference_component(
    InferenceComponentName=finqa_ic_name,
    EndpointName=endpoint_name,
    Specification={
        "BaseInferenceComponentName": base_ic_name,
        "Container": {
            "ArtifactUrl": finqa_adapter_s3_uri,
        },
    },
)

wait_for_ic(finqa_ic_name)


### C.5 Test the FinQA adapter IC


In [ ]:
finqa_payload = json.dumps({
    "messages": [
        {"role": "system", "content": FINQA_SYSTEM_PROMPT},
        {"role": "user",   "content": (
            "Please answer the following financial question based on the context provided. "
            "Make sure to give the answer in raw number, not in percentage and without any units. "
            "If the question asks for percentage the value should be less than 1\n"
            "Context:\n"
            "net revenues increased $ 4.2 billion , or 11% , to $ 42.3 billion in 2007 compared "
            "with $ 38.1 billion in 2006 . net revenues increased $ 4.9 billion , or 15% , to "
            "$ 38.1 billion in 2006 compared with $ 33.2 billion in 2005 .\n"
            "what was the net revenue in 2005?"
        )},
    ],
    "max_tokens": 256,
    "temperature": 0.1,
})

t0 = time.time()
res_finqa = core_endpoint.invoke(
    body=finqa_payload,
    content_type="application/json",
    accept="application/json",
    inference_component_name=finqa_ic_name,
)
response_finqa = json.loads(res_finqa.body.read().decode("utf-8"))
print(f"Response time: {time.time()-t0:.2f}s")
display(Markdown(response_finqa["choices"][0]["message"]["content"]))
print(f"\nUsage: {response_finqa['usage']}")


### C.6 Three-way comparison: base vs medical adapter vs FinQA adapter

Send the same financial question to all three ICs to see how specialisation affects the response.


In [ ]:
for label, ic, resp_obj in [
    ("BASE MODEL",       base_ic_name,    None),
    ("MEDICAL ADAPTER",  adapter_ic_name, None),
    ("FINQA ADAPTER",    finqa_ic_name,   None),
]:
    res = core_endpoint.invoke(
        body=finqa_payload,
        content_type="application/json",
        accept="application/json",
        inference_component_name=ic,
    )
    answer = json.loads(res.body.read().decode("utf-8"))["choices"][0]["message"]["content"]
    print(f"{'=' * 60}")
    print(f"{label}")
    print(f"{'=' * 60}")
    print(answer[:600])
    print()


---
## Store variables for downstream notebooks

Saves the endpoint name and all Inference Component names to the IPython store so
`03.02_benchmark_ic_vs_non_ic.ipynb` can load them automatically with `%store -r`.

| Variable | Description |
|---|---|
| `IC_ENDPOINT_NAME` | The IC-enabled endpoint created in Part A |
| `BASE_IC_NAME` | Base Qwen3-4B Inference Component (Part A) |
| `ADAPTER_IC_NAME` | Medical reasoning LoRA adapter IC (Part B) |
| `FINQA_IC_NAME` | Financial reasoning LoRA adapter IC (Part C) |

In [ ]:
# Store endpoint and IC names for use in 03.02 benchmark notebook
IC_ENDPOINT_NAME = endpoint_name
BASE_IC_NAME     = base_ic_name
ADAPTER_IC_NAME  = adapter_ic_name
FINQA_IC_NAME    = finqa_ic_name

%store IC_ENDPOINT_NAME
%store BASE_IC_NAME
%store ADAPTER_IC_NAME
%store FINQA_IC_NAME

print("Stored variables:")
print(f"  IC_ENDPOINT_NAME : {IC_ENDPOINT_NAME}")
print(f"  BASE_IC_NAME     : {BASE_IC_NAME}")
print(f"  ADAPTER_IC_NAME  : {ADAPTER_IC_NAME}")
print(f"  FINQA_IC_NAME    : {FINQA_IC_NAME}")
print("\nThese are available in 03.02 via %store -r")

---
## Cleanup

Delete all resources created in this notebook to avoid ongoing charges.

In [ ]:
RUN_CLEANUP = False  # Set to True to delete all resources

if RUN_CLEANUP:
    # Delete all Inference Components first
    for ic in [finqa_ic_name, adapter_ic_name, base_ic_name]:
        try:
            sm_client.delete_inference_component(InferenceComponentName=ic)
            print(f"Deleted IC: {ic}")
        except Exception as e:
            print(f"Could not delete IC {ic}: {e}")

    time.sleep(10)  # allow ICs to finish deleting before removing the endpoint


In [ ]:
if RUN_CLEANUP:
    # Delete endpoint, endpoint config, and model using SDK v3 resource objects
    try:
        Endpoint.get(endpoint_name=endpoint_name).delete()
        print(f"Deleted endpoint: {endpoint_name}")
    except Exception as e:
        print(f"Could not delete endpoint: {e}")

    try:
        EndpointConfig.get(endpoint_config_name=endpoint_config_name).delete()
        print(f"Deleted endpoint config: {endpoint_config_name}")
    except Exception as e:
        print(f"Could not delete endpoint config: {e}")

    try:
        Model.get(model_name=model_name).delete()
        print(f"Deleted model: {model_name}")
    except Exception as e:
        print(f"Could not delete model: {e}")
